In [3]:
################################################################################
#  NOTEBOOK COLAB — MODELO MILP DE COMPRA DE IMPORTADOS · IKEA FOOD CHILE
#  Tesis MOL · UNAB · Francisco González
#
#  Ejecuta el cálculo del PEDIDO DEL MES con datos reales de BigQuery.
#  Formato: pega cada celda respetando su tipo (MARKDOWN / CÓDIGO).
#  Separador de celda:  === CELDA n [TIPO] ===        Fin de celda:  ---
################################################################################

# Modelo MILP de compra — IKEA Food Chile

"""Calcula **el pedido de este mes** para los productos importados.

**Proceso que replica:** cada mes el equipo comercial entrega un forecast a 12 meses;
el modelo calcula el pedido a poner **hoy**, que arribará en ~6 meses (liberación SEREMI).

**Qué optimiza:** minimiza el inventario (pallet-mes) **sujeto a** mantener el nivel de
servicio. Como la tarifa de bodega es la misma para toda posición de pallet, minimizar
pallet-mes equivale a minimizar el gasto de bodega, sin necesidad de calibrar costos.

**Entrega:** dos sugerencias de compra (vida útil BASE y CONSERVADORA) en un Excel con
pallets, cajas, unidad base y meses de venta, ajustable antes de cargar al proveedor.


SyntaxError: incomplete input (1785297121.py, line 12)

In [4]:

# ============================================================================
#  CONFIGURACIÓN — revisar antes de cada corrida
# ============================================================================
PROJECT   = "tc-sc-bi-bigdata-ikea-corp-qa"
ANCHOR    = "2026-07-01"     # primer mes del horizonte (mes en curso)
T         = 12               # horizonte del forecast (meses)
L         = 6                # lead time: orden → liberación SEREMI (se RECALCULA de datos)
BIAS      = 0.232            # sesgo del forecast al lag 6 (sección 8)
UF_CLP    = 40_844.79        # ⚠️ valor UF del día (cambia a diario)
TARIFA_UF_PALLET_DIA = 0.3019
IVA       = 1.19

HIST_INI  = "2024-07-01"     # ventana histórica (ABC y σ)
HIST_FIN  = "2026-06-30"
DIAS_MES  = 30.44

# Rutas de tablas
TBL_MAESTRO = f"`{PROJECT}.Availability_FOOD_CL.Equivalencias Importados`"
TBL_IMPORT  = f"`{PROJECT}.Availability_FOOD_CL.Seguimiento_Importaciones`"
TBL_FCST    = f"`{PROJECT}.Availability_FOOD_CL.Hist Fcst mensual`"
TBL_BOM     = f"`{PROJECT}.Availability_FOOD_CL.BOM_df`"
TBL_VENC    = f"`{PROJECT}.Availability_FOOD_CL.vencimiento_producto_mes`"
TBL_INV     = f"`{PROJECT}.Availability_FOOD_CL.fact_inventory_projection_12m_integrated`"
TBL_ITEMS   = f"`{PROJECT}.Availability_FOOD_CL.items`"
TBL_RECETAS = f"`{PROJECT}.Availability_FOOD_CL.Recipe_list_cl`"
TBL_BINS    = "`ftc-ikso-analytics-work-wbx.corp_proc_schedule_mvbc.raw_retail_bin_entries_cl`"

# Presupuesto real de bodega (línea base para valorizar el ahorro)
PRESUPUESTO_BODEGA = {
    "072026": 11_615_825.4, "082026": 12_225_707.8, "092026": 17_436_562.1,
    "102026": 18_246_614.4, "112026": 15_654_916.6, "122026": 12_530_059.6,
    "012027": 11_571_544.6, "022027": 11_709_948.0, "032027": 11_711_938.3,
    "042027": 11_536_444.6, "052027": 12_468_673.9, "062027": 11_806_994.9,
}
GASTO_BASE_ANUAL = sum(PRESUPUESTO_BODEGA.values())   # $158.515.230

In [5]:
# === CELDA 3 — Setup: conexión y modelo ===
!pip -q install pulp

import sys, os, shutil, glob

# 1) Trae el repo (si ya está clonado, no pasa nada)
!git clone -q https://github.com/PanchoGonzaMo/Supply-Chain-for-foods 2>/dev/null || true

# 2) Localiza el modelo correcto (ignora la versión vieja _Importados_IKEA)
candidatos = [f for f in glob.glob("**/modelo_milp_compra*.py", recursive=True)
              if "Importados_IKEA" not in f]
assert candidatos, "No encuentro modelo_milp_compra*.py — súbelo al repo o a /content"
origen = candidatos[0]

# 3) Cópialo a /content con el nombre exacto e importable (sin espacios ni sufijos)
shutil.copy(origen, "/content/modelo_milp_compra.py")
if "/content" not in sys.path:
    sys.path.insert(0, "/content")

# 4) Limpia imports previos fallidos (por si corriste la celda antes)
for m in list(sys.modules):
    if "modelo_milp_compra" in m:
        del sys.modules[m]
print(f"✓ modelo cargado desde: {origen}")

# 5) Conexión a BigQuery
from google.colab import auth; auth.authenticate_user()
from google.cloud import bigquery
import pandas as pd, numpy as np

client = bigquery.Client(project=PROJECT)
def bq_query(sql: str) -> pd.DataFrame:
    """REST (evita el error de permisos del BigQuery Storage API)."""
    return client.query(sql).result().to_dataframe(create_bqstorage_client=False)

# 6) Importa el modelo
from modelo_milp_compra import (
    SKU, ModelConfig, solve_milp, solve_scenarios, pedido_del_mes,
    shelf_life_from_imports, extend_demand, export_excel, plot_sku,
)
print("✓ listo")

✓ modelo cargado desde: Supply-Chain-for-foods/modelo_milp_compra.py
✓ listo


In [ ]:
###=== CELDA 4 [MARKDOWN] ===###
## A · Maestro de productos
"""`units_per_pallet` (CAP_p), `units_per_case` y el **grupo de contenedor** (origen × tipo),
que define qué SKU pueden compartir embarque.

Unidades por caja: si `UoM_Base='PC'` → `Q_x_EMB`; si `='KG'` → `Q_x_EMB × KG_x_UN`.


In [6]:
###=== CELDA 5 [CÓDIGO] ===###
maestro = bq_query(f"""
SELECT CAST(Codigo_MVBC AS STRING) AS sku,
       UPPER(TRIM(UoM_Base)) AS uom_base,
       SAFE_CAST(Q_x_EMB AS FLOAT64) AS q_x_emb,
       SAFE_CAST(KG_x_UN AS FLOAT64) AS kg_x_un,
       SAFE_CAST(UV_Pallet_KG_Pallet AS FLOAT64) AS units_per_pallet,
       Descripcion_MVBC AS descripcion,
       CAST(HFB AS STRING) AS hfb,
       UPPER(TRIM(Tipo_producto)) AS tipo_producto,
       UPPER(TRIM(Origen)) AS origen
FROM {TBL_MAESTRO} WHERE Codigo_MVBC IS NOT NULL
""")
maestro["units_per_case"] = np.where(maestro.uom_base == "PC",
                                     maestro.q_x_emb,
                                     maestro.q_x_emb * maestro.kg_x_un)
maestro["container_group"] = maestro.origen + "-" + maestro.tipo_producto

mal = maestro[(maestro.units_per_pallet.isna()) | (maestro.units_per_pallet <= 0) |
              (maestro.units_per_case.isna())   | (maestro.units_per_case <= 0)]
if len(mal):
    print(f"⚠️ {len(mal)} SKU con pallet/caja inválido — excluidos")
    display(mal[["sku","descripcion","uom_base","q_x_emb","kg_x_un","units_per_pallet"]].head())
maestro = maestro.drop(index=mal.index)
importados = set(maestro.sku)
print(f"✓ Maestro: {len(maestro)} SKU")
print(maestro.groupby("container_group").size().to_string())



⚠️ 37 SKU con pallet/caja inválido — excluidos


,sku,descripcion,uom_base,q_x_emb,kg_x_un,units_per_pallet
7,789000067,Soda and juice cup lids,KG,1000.0,0.006000,NaN
8,789000069,Soda and juice cup,KG,1000.0,0.011500,NaN
9,789000068,Coffee and tea cup sleeve,KG,1000.0,0.004700,NaN
10,789000063,Cup holder for coffee (2 units),KG,600.0,0.011833,NaN
11,789000064,Cup holder for coffee (4 units),KG,300.0,0.026667,NaN


✓ Maestro: 138 SKU
container_group
SE-CONGELADO    67
SE-SECO         71


In [ ]:
###=== CELDA 6 [MARKDOWN] ===###
## B · Importaciones: lead time, vida útil y arribos
"""Tres insumos de una sola tabla:
1. **Lead time real** (`Fecha_Liberacion_Real − Fecha_Orden`) — se **mide**, no se asume.
2. **Vida útil residual** al arribo → dos escenarios (BASE ponderado / CONSERVADOR P25).
3. **Arribos confirmados** (en tránsito / preparación) — ya comprometidos, no se re-piden.

Se usan `Arribado` **y** `En transito` para la vida útil: en tránsito ya trae el BBD real
del proveedor. Solo `En preparacion` queda fuera.
---


In [7]:
###=== CELDA 7 [CÓDIGO] ===###
FMT_FECHA = '%d-%m-%Y'      # ⚠️ ajustar a '%m-%d-%Y' si el diagnóstico lo indica

imp = bq_query(f"""
SELECT CAST(Codigo_MVBC AS STRING) AS sku,
       UPPER(TRIM(Estado_Importacion)) AS estado,
       SAFE_CAST(Cantidad_UoM_Base AS FLOAT64) AS cantidad_u,
       SAFE_CAST(Orden_E_shop AS STRING) AS orden,
       SAFE.PARSE_DATE('{FMT_FECHA}', TRIM(CAST(Fecha_Orden AS STRING)))            AS fecha_orden,
       SAFE.PARSE_DATE('{FMT_FECHA}', TRIM(CAST(Fecha_Liberacion_Real AS STRING)))  AS fecha_liberacion,
       SAFE.PARSE_DATE('{FMT_FECHA}', TRIM(CAST(BBD AS STRING)))                    AS bbd,
       SAFE_CAST(Vida_util_liberacion AS FLOAT64) AS vu_dias
FROM {TBL_IMPORT}
WHERE Codigo_MVBC IS NOT NULL AND SAFE_CAST(Cantidad_UoM_Base AS FLOAT64) > 0
""")
for c in ["fecha_orden","fecha_liberacion","bbd"]:
    imp[c] = pd.to_datetime(imp[c])

# --- Control de parseo: si muchas fechas quedan nulas, el formato está mal ---
tot = len(imp)
for c in ["fecha_orden","fecha_liberacion","bbd"]:
    nulos = imp[c].isna().sum()
    print(f"  {c:18s}: {nulos:>5,} nulos de {tot:,} ({nulos/tot:.1%})")
if imp["fecha_orden"].isna().mean() > 0.10:
    print("⚠️  Demasiadas fechas de orden sin parsear → revisar FMT_FECHA")

print()
print(imp.estado.value_counts().to_string())


  fecha_orden       :   427 nulos de 441 (96.8%)
  fecha_liberacion  :   441 nulos de 441 (100.0%)
  bbd               :   441 nulos de 441 (100.0%)
⚠️  Demasiadas fechas de orden sin parsear → revisar FMT_FECHA

estado
ARRIBADO          404
EN PREPARACION     37


In [8]:
###=== CELDA 7 [CÓDIGO] ===###
# Todas las columnas son STRING. Las fechas vienen en formatos mezclados
# (ISO con y sin hora, y algunas DD-MM-YYYY), por eso el parseo es tolerante.
def _fecha(col):
    return f"""COALESCE(
      SAFE.PARSE_DATE('%Y-%m-%d', SUBSTR(TRIM(CAST({col} AS STRING)),1,10)),
      SAFE.PARSE_DATE('%d-%m-%Y', SUBSTR(TRIM(CAST({col} AS STRING)),1,10)),
      SAFE.PARSE_DATE('%d/%m/%Y', SUBSTR(TRIM(CAST({col} AS STRING)),1,10))
    )"""
def _num(col):
    # numéricos como STRING: quita separador de miles y normaliza decimal
    return f"SAFE_CAST(REPLACE(REPLACE(TRIM(CAST({col} AS STRING)),'.',''),',','.') AS FLOAT64)"

imp = bq_query(f"""
SELECT CAST(Codigo_MVBC AS STRING)              AS sku,
       UPPER(TRIM(Estado_Importacion))          AS estado,
       UPPER(TRIM(Origen))                      AS origen,
       UPPER(TRIM(Tipo))                        AS tipo,
       SAFE_CAST(TRIM(Cantidad_UoM_Base) AS FLOAT64) AS cantidad_u,
       CAST(Orden_E_shop AS STRING)             AS orden,
       {_fecha('Fecha_Orden')}                  AS fecha_orden,
       {_fecha('Fecha_Liberacion_Real')}        AS fecha_liberacion,
       {_fecha('Fecha_Liberacion_Estimada')}    AS fecha_liberacion_est,
       {_fecha('ETA')}                          AS eta,
       {_fecha('BBD')}                          AS bbd,
       {_fecha('Production_Date')}              AS production_date,
       SAFE_CAST(TRIM(Vida_util_liberacion) AS FLOAT64) AS vu_dias,
       -- lead time YA CALCULADO por la fuente, y su desglose por etapa
       SAFE_CAST(TRIM(Total_Orden_dias) AS FLOAT64)  AS lt_total_dias,
       SAFE_CAST(TRIM(Transito) AS FLOAT64)          AS et_transito,
       SAFE_CAST(TRIM(Internacion) AS FLOAT64)       AS et_internacion,
       SAFE_CAST(TRIM(Liberacion) AS FLOAT64)        AS et_liberacion,
       SAFE_CAST(TRIM(Picking_y_etiquetado) AS FLOAT64) AS et_picking,
       SAFE_CAST(TRIM(Dias_Seremi) AS FLOAT64)       AS dias_seremi
FROM {TBL_IMPORT}
WHERE Codigo_MVBC IS NOT NULL
  AND SAFE_CAST(TRIM(Cantidad_UoM_Base) AS FLOAT64) > 0
""")
for c in ["fecha_orden","fecha_liberacion","fecha_liberacion_est","eta","bbd","production_date"]:
    imp[c] = pd.to_datetime(imp[c])

print(f"Registros: {len(imp):,}")
print(imp.estado.value_counts().to_string())
print("\nCompletitud:")
for c in ["fecha_orden","fecha_liberacion","bbd","vu_dias","lt_total_dias"]:
    n=imp[c].isna().sum(); print(f"  {c:22s} {n:>4,} nulos ({n/len(imp):.0%})")

Registros: 441
estado
ARRIBADO          404
EN PREPARACION     37

Completitud:
  fecha_orden               0 nulos (0%)
  fecha_liberacion          0 nulos (0%)
  bbd                      29 nulos (7%)
  vu_dias                  29 nulos (7%)
  lt_total_dias             0 nulos (0%)


In [9]:


##=== CELDA 7.1 [CÓDIGO] — LEAD TIME (solo órdenes de planificación) ===##
# ⚠️ CAMBIO: el lead time se calcula SOLO con las órdenes cuyo Orden_E_shop
# empieza con "4". Las demás corresponden a actividades comerciales puntuales
# (lanzamientos, campañas) que no siguen la planificación habitual de compra y
# distorsionan el promedio.
#
# ⚠️ CAMBIO: la orden se emite ~1 semana DESPUÉS del cálculo del modelo, de modo
# que el arribo ocurre a los (7 + lead_time) días de la corrida.
DIAS_EMISION_PEDIDO = 7      # días entre el cálculo del modelo y la emisión real

imp["orden_planificacion"] = imp.orden.astype(str).str.strip().str.startswith("4")
print("Órdenes por tipo:")
print(f"  planificación (empiezan con 4): {imp.orden_planificacion.sum():>5,}")
print(f"  comerciales / otras           : {(~imp.orden_planificacion).sum():>5,}")

# --- Fuente A: campo Total_Orden_dias ya calculado ---
ltA = imp[imp.orden_planificacion & imp.lt_total_dias.notna()
          & (imp.lt_total_dias > 0) & (imp.lt_total_dias < 720)]
# --- Fuente B: diferencia de fechas ---
ltB = imp[imp.orden_planificacion & imp.fecha_liberacion.notna()
          & imp.fecha_orden.notna()].copy()
ltB["lt_dias"] = (ltB.fecha_liberacion - ltB.fecha_orden).dt.days
ltB = ltB[(ltB.lt_dias > 0) & (ltB.lt_dias < 720)]

print("\nLEAD TIME — solo órdenes de planificación")
if len(ltA):
    a = np.average(ltA.lt_total_dias, weights=ltA.cantidad_u)
    print(f"  A · Total_Orden_dias : {a:,.0f} días = {a/DIAS_MES:.1f} meses (n={len(ltA)})")
if len(ltB):
    b = np.average(ltB.lt_dias, weights=ltB.cantidad_u)
    print(f"  B · dif. de fechas   : {b:,.0f} días = {b/DIAS_MES:.1f} meses (n={len(ltB)})")
if len(ltA) and len(ltB):
    print(f"  → discrepancia: {abs(a-b):,.0f} días {'✓' if abs(a-b)<10 else '⚠️ revisar'}")

# Comparación contra las órdenes comerciales (justifica el filtro en la tesis)
com = imp[~imp.orden_planificacion & imp.lt_total_dias.notna() & (imp.lt_total_dias > 0)]
if len(com):
    print(f"\n  Órdenes comerciales (excluidas): {com.lt_total_dias.mean():,.0f} días promedio "
          f"(n={len(com)}) — distinto régimen, por eso se separan")

lt = ltB if len(ltB) >= len(ltA) else ltA.assign(lt_dias=ltA.lt_total_dias)
lt_prom = np.average(lt.lt_dias, weights=lt.cantidad_u)

# Lead time EFECTIVO = emisión del pedido + tránsito/liberación
LT_EFECTIVO_DIAS = DIAS_EMISION_PEDIDO + lt_prom
L = int(round(LT_EFECTIVO_DIAS / DIAS_MES))
print(f"\n  lead time logístico  : {lt_prom:,.0f} días")
print(f"  + emisión del pedido : {DIAS_EMISION_PEDIDO} días")
print(f"  = LEAD TIME EFECTIVO : {LT_EFECTIVO_DIAS:,.0f} días = {LT_EFECTIVO_DIAS/DIAS_MES:.2f} meses")
print(f"  σ_L = {lt.lt_dias.std():,.0f} días ({lt.lt_dias.std()/DIAS_MES:.1f} meses)")
print(f"  → L usado en el modelo: {L} meses")

# Variabilidad por grupo de consolidación
ltg = lt.merge(maestro[["sku","container_group"]], on="sku", how="left")
print("\nPor grupo (origen × tipo):")
print(ltg.groupby("container_group").lt_dias.agg(n="count", media="mean", sd="std")
         .round(0).to_string())

# Desglose por etapa (alimenta la Figura 6 de la tesis)
etapas = ["et_transito","et_internacion","et_liberacion","et_picking"]
disp = [c for c in etapas if c in imp.columns and imp[c].notna().any()]
if disp:
    print("\nDesglose del lead time por etapa (días promedio, órdenes de planificación):")
    print(imp[imp.orden_planificacion][disp].mean().round(1).to_string())

Órdenes por tipo:
  planificación (empiezan con 4):   411
  comerciales / otras           :    30

LEAD TIME — solo órdenes de planificación
  A · Total_Orden_dias : 175 días = 5.7 meses (n=411)
  B · dif. de fechas   : 175 días = 5.7 meses (n=411)
  → discrepancia: 0 días ✓

  Órdenes comerciales (excluidas): 176 días promedio (n=30) — distinto régimen, por eso se separan

  lead time logístico  : 175 días
  + emisión del pedido : 7 días
  = LEAD TIME EFECTIVO : 182 días = 5.96 meses
  σ_L = 26 días (0.9 meses)
  → L usado en el modelo: 6 meses

Por grupo (origen × tipo):
                   n  media    sd
container_group                  
SE-CONGELADO     476  174.0  28.0
SE-SECO          541  179.0  24.0

Desglose del lead time por etapa (días promedio, órdenes de planificación):
et_transito       45.4
et_internacion     7.7
et_liberacion     28.7
et_picking        93.4


In [10]:
###=== CELDA 7.2 — VIDA ÚTIL RESIDUAL (dos escenarios) ===###
# Solo estados con BBD confiable. Antes asumíamos incluir "EN TRANSITO",
# pero en estos datos ese estado no existe: quedan ARRIBADO y EN PREPARACION.
vu = imp[imp.estado.str.contains("ARRIB", na=False)].copy()

# vida útil en meses: usa el campo directo; si falta, la deriva del BBD
vu["arribo_ref"] = vu.fecha_liberacion.fillna(vu.eta)
vu["vida_util_restante_meses"] = np.where(
    vu.vu_dias.notna(), vu.vu_dias / DIAS_MES,
    (vu.bbd - vu.arribo_ref).dt.days / DIAS_MES)
vu = vu[vu.vida_util_restante_meses > 0].rename(columns={"cantidad_u":"unidades_arribadas"})

shelf = shelf_life_from_imports(
    vu[["sku","vida_util_restante_meses","unidades_arribadas"]])

print(f"✓ Vida útil para {len(shelf)} SKU (de {imp.sku.nunique()} en importaciones)")
print(shelf[["sku","vu_prom_pond","vu_p25","shelf_life_w","shelf_life_p25","n_embarques"]]
      .head(10).to_string(index=False))

pocos = shelf[shelf.n_embarques < 3]
if len(pocos):
    print(f"\n⚠️  {len(pocos)} SKU con <3 embarques: su P25 no es robusto.")

✓ Vida útil para 70 SKU (de 71 en importaciones)
      sku  vu_prom_pond  vu_p25  shelf_life_w  shelf_life_p25  n_embarques
789000000         13.88    6.34            13               6            5
789000003         16.43   16.29            16              16            6
789000004          7.49    4.57             7               4            9
789000005         12.94   12.42            12              12            5
789000010          5.99    4.76             5               4           10
789000011          6.40    5.52             6               5           10
789000012         13.00   12.91            12              12            9
789000014          7.33    6.57             7               6            8
789000015          6.22    6.24             6               6            6
789000016          6.18    4.11             6               4            4

⚠️  19 SKU con <3 embarques: su P25 no es robusto.


In [11]:
###=== CELDA 7.3 — ARRIBOS CONFIRMADOS A_{p,t} ===###
# Compras ya comprometidas: el modelo NO debe volver a pedirlas, solo sumarlas.
pend = imp[imp.estado.str.contains("TRANSITO|PREPARAC", na=False, regex=True)].copy()

# Mejora: ahora existe Fecha_Liberacion_Estimada, mejor que estimar orden + lead time.
pend["fecha_arribo"] = (pend.fecha_liberacion
                        .fillna(pend.fecha_liberacion_est)
                        .fillna(pend.eta)
                        .fillna(pend.fecha_orden + pd.to_timedelta(lt_prom, "D")))
pend["mes"] = pend.fecha_arribo.dt.to_period("M").dt.to_timestamp()

arr_long = (pend.groupby(["sku","mes"], as_index=False).cantidad_u.sum()
                .rename(columns={"cantidad_u":"arr_u"}))

print(f"✓ Arribos confirmados: {len(pend)} embarques · {pend.cantidad_u.sum():,.0f} u")
print(f"  fuente de la fecha: real={pend.fecha_liberacion.notna().sum()}, "
      f"estimada={pend.fecha_liberacion_est.notna().sum()}, "
      f"ETA={pend.eta.notna().sum()}")
print("\nPor mes de arribo:")
print(arr_long.groupby("mes").arr_u.sum().head(12).to_string())

✓ Arribos confirmados: 37 embarques · 142,495 u
  fuente de la fecha: real=37, estimada=37, ETA=37

Por mes de arribo:
mes
2026-09-01    59616.0
2026-10-01    82879.2


In [ ]:
###=== CELDA 8 [MARKDOWN] ===###
## C · Forecast de recetas → demanda de importados (explosión RECURSIVA)
"""El forecast está a nivel de **receta** (`598…`). Los productos comprados son `789…`.
La explosión debe ser **recursiva**: `receta → sub-receta (598) → producto (789)`.
Una explosión de un solo nivel **pierde** los importados que viven dentro de una sub-receta.

De la misma tabla sale el **lag** = `mes_fcst − mes_calculo`, que alimenta el σ (celda E).
---

In [12]:
###=== CELDA 9 [CÓDIGO] ===###
SQL_EXPLOSION = """
WITH RECURSIVE
bom AS (
  SELECT CAST(Position AS STRING) AS padre, CAST(`No` AS STRING) AS componente,
         SAFE_CAST(Quantity_per AS FLOAT64) AS qty_per,
         UPPER(TRIM(BOM_Unit_of_Measure_Code)) AS bom_uom,
         UPPER(TRIM(Base_UoM)) AS base_uom
  FROM {BOM} WHERE SAFE_CAST(Quantity_per AS FLOAT64) IS NOT NULL
),
raiz AS ({RAIZ}),
exp AS (
  SELECT r.*, b.componente,
         CASE WHEN b.bom_uom='GR' AND b.base_uom='KG' THEN r.cantidad*b.qty_per/1000
              WHEN b.bom_uom=b.base_uom THEN r.cantidad*b.qty_per
              ELSE NULL END AS necesidad, 1 AS nivel
  FROM raiz r JOIN bom b ON b.padre = r.receta
  UNION ALL
  SELECT {COLS}, b.componente,
         CASE WHEN b.bom_uom='GR' AND b.base_uom='KG' THEN e.necesidad*b.qty_per/1000
              WHEN b.bom_uom=b.base_uom THEN e.necesidad*b.qty_per
              ELSE NULL END, e.nivel+1
  FROM exp e JOIN bom b ON b.padre = e.componente
  WHERE STARTS_WITH(e.componente,'598') AND e.necesidad IS NOT NULL AND e.nivel < 5
)
SELECT {SEL}, componente AS sku, SUM(necesidad) AS cantidad_final
FROM exp
WHERE necesidad IS NOT NULL AND NOT STARTS_WITH(componente,'598')
GROUP BY {GRP}, sku
"""

sql_fcst = SQL_EXPLOSION.format(
    BOM=TBL_BOM,
    RAIZ=f"""SELECT CAST(Item_No AS STRING) AS receta,
                    SAFE_CAST(Ano_calculo AS INT64) AS ano_c,
                    SAFE_CAST(Mes_Calculo AS INT64) AS mes_c,
                    SAFE_CAST(Ano_Fcst AS INT64) AS ano_f,
                    SAFE_CAST(Mes_Fcst AS INT64) AS mes_f,
                    SUM(SAFE_CAST(Fcst AS FLOAT64)) AS cantidad
             FROM {TBL_FCST} WHERE Fcst IS NOT NULL
             GROUP BY receta, ano_c, mes_c, ano_f, mes_f""",
    COLS="e.receta, e.ano_c, e.mes_c, e.ano_f, e.mes_f, e.cantidad",
    SEL="ano_c, mes_c, ano_f, mes_f, (ano_f*12+mes_f)-(ano_c*12+mes_c) AS lag_meses, DATE(ano_f,mes_f,1) AS mes_objetivo",
    GRP="ano_c, mes_c, ano_f, mes_f, lag_meses, mes_objetivo")

fcst_bom = bq_query(sql_fcst)
fcst_bom["mes_objetivo"] = pd.to_datetime(fcst_bom.mes_objetivo)
fcst_bom = fcst_bom[fcst_bom.sku.isin(importados)].rename(columns={"cantidad_final":"demanda_u"})
print(f"✓ Forecast explotado: {fcst_bom.sku.nunique()} SKU · lags {fcst_bom.lag_meses.min()}–{fcst_bom.lag_meses.max()}")

# Forecast VIGENTE: última corrida de cálculo, meses del horizonte
ult = (fcst_bom.ano_c*12 + fcst_bom.mes_c).max()
anchor_ts = pd.Timestamp(ANCHOR)
vig = fcst_bom[((fcst_bom.ano_c*12+fcst_bom.mes_c) == ult) &
               (fcst_bom.mes_objetivo >= anchor_ts) &
               (fcst_bom.mes_objetivo <  anchor_ts + pd.DateOffset(months=T))]
fcst_long = vig.groupby(["sku","mes_objetivo"], as_index=False).demanda_u.sum().rename(
    columns={"mes_objetivo":"mes","demanda_u":"forecast_u"})
print(f"✓ Forecast vigente: {fcst_long.sku.nunique()} SKU × {fcst_long.mes.nunique()} meses")

# Snapshots a lag = L (para el σ)
snap = fcst_bom[fcst_bom.lag_meses == L][["sku","mes_objetivo","demanda_u"]].rename(
    columns={"demanda_u":"fcst_u"})
print(f"✓ Snapshots a lag {L}: {len(snap):,} filas")


✓ Forecast explotado: 60 SKU · lags 0–17
✓ Forecast vigente: 55 SKU × 12 meses
✓ Snapshots a lag 6: 1,114 filas


In [ ]:
###=== CELDA 10 [MARKDOWN] ===###
## E · Consumo real, σ (RMSE) y clasificación ABC
"""El consumo sale de las **salidas de inventario** (`item_no` ya es la materia prima; refleja
el BOM vigente al momento de la venta, no el actual).

`entry_type='Sale'` (venta directa) o `'Negative Adjmt.' + reason_code='IFB'` (consumo por
receta). Ese `reason_code` **aísla el consumo de las mermas**: sin él, el σ mediría el error
del pronóstico contra movimientos que el forecast nunca intentó predecir.

In [13]:
###=== CELDA 11 [CÓDIGO] — σ DESESGADO (reemplaza el cálculo anterior) ===###
# ⚠️ CAMBIO METODOLÓGICO IMPORTANTE
#
# Antes: σ = RMSE = √( promedio( (real − forecast_crudo)² ) )
#
# El problema: RMSE² = sesgo² + varianza. Como el modelo YA corrige el sesgo
# aparte (divide el forecast por 1+bias), usar el RMSE crudo como σ contabiliza
# el sesgo DOS VECES e infla el stock de seguridad.
#
# Ahora: σ = desviación estándar de los residuos DESPUÉS de desesgar. Mide solo
# la dispersión —que es lo que el stock de seguridad debe cubrir—, no el error
# sistemático, que se corrige en la demanda.
ventas_long = bq_query(f"""
SELECT CAST(item_no AS STRING) AS sku,
       DATE_TRUNC(DATE(posting_date), MONTH) AS mes,
       SUM(ABS(SAFE_CAST(quantity AS FLOAT64))) AS venta_u
FROM {TBL_BINS}
WHERE DATE(posting_date) BETWEEN '{HIST_INI}' AND '{HIST_FIN}'
  AND location_code IN ('594_IFB','636_IFB')
  AND SAFE_CAST(quantity AS FLOAT64) < 0
  AND (TRIM(entry_type)='Sale'
    OR (TRIM(entry_type)='Negative Adjmt.' AND TRIM(reason_code)='IFB'))
GROUP BY sku, mes
""")
ventas_long["mes"] = pd.to_datetime(ventas_long.mes)
ventas_long = ventas_long[ventas_long.sku.isin(importados)]
print(f"✓ Consumo real: {ventas_long.sku.nunique()} SKU · {ventas_long.venta_u.sum():,.0f} u")

# --- Pares forecast(lag L) vs. consumo real ---
pares = snap.merge(ventas_long, left_on=["sku","mes_objetivo"], right_on=["sku","mes"])

# 1) SESGO por SKU (multiplicativo): cuánto sobre/sub-pronostica
sesgo_sku = (pares.groupby("sku")
             .apply(lambda g: (g.fcst_u.sum()/g.venta_u.sum() - 1) if g.venta_u.sum()>0 else np.nan)
             .rename("sesgo").reset_index())
sesgo_global = (pares.fcst_u.sum()/pares.venta_u.sum() - 1) if pares.venta_u.sum()>0 else BIAS
print(f"\nSesgo agregado al lag {L}: {sesgo_global:+.1%}  (sección 10 reporta +23,2 %)")

# 2) RESIDUOS DESESGADOS: se corrige el forecast antes de medir la dispersión
pares = pares.merge(sesgo_sku, on="sku", how="left")
pares["sesgo_aplicado"] = pares.sesgo.fillna(sesgo_global).clip(-0.5, 2.0)
pares["fcst_corregido"] = pares.fcst_u / (1 + pares.sesgo_aplicado)
pares["resid"] = pares.venta_u - pares.fcst_corregido

# 3) σ = desviación estándar de los residuos (NO el RMSE crudo)
sig = (pares.groupby("sku")
       .agg(sigma=("resid","std"),
            rmse_crudo=("resid", lambda x: float(np.sqrt((x**2).mean()))),
            n_meses=("resid","size"),
            venta_prom=("venta_u","mean")).reset_index())
sig = sig[(sig.n_meses >= 6) & sig.sigma.notna()]
print(f"σ propio calculado para {len(sig)} SKU (≥6 meses de historia)")

# Comparación: cuánto bajó el σ al desesgar
comp_rmse = (pares.groupby("sku")
             .apply(lambda g: float(np.sqrt(((g.venta_u-g.fcst_u)**2).mean())))
             .rename("rmse_con_sesgo").reset_index())
cmpx = sig.merge(comp_rmse, on="sku")
if len(cmpx):
    red = 1 - (cmpx.sigma.sum()/cmpx.rmse_con_sesgo.sum())
    print(f"  σ desesgado vs. RMSE crudo: reducción de {red:.0%} en la escala del SS")

# --- Fallback para SKU sin historia suficiente ---
# Se usa un σ RELATIVO (coeficiente de variación mediano), no un valor absoluto:
# un RMSE agregado nacional no tiene sentido aplicado a un SKU de baja rotación.
cv_mediano = float((sig.sigma / sig.venta_prom.replace(0, np.nan)).median())
if not np.isfinite(cv_mediano) or cv_mediano <= 0:
    cv_mediano = 0.35
print(f"  CV mediano (σ/venta) = {cv_mediano:.2f} → base del fallback")

vprom = ventas_long.groupby("sku", as_index=False).venta_u.mean().rename(
    columns={"venta_u":"venta_prom_sku"})
sigma_full = (maestro[["sku","hfb"]]
              .merge(sig[["sku","sigma","n_meses"]], on="sku", how="left")
              .merge(vprom, on="sku", how="left"))
falta = sigma_full.sigma.isna()
sigma_full.loc[falta,"sigma"] = (sigma_full.loc[falta,"venta_prom_sku"].fillna(0) * cv_mediano)
sigma_full["sigma"] = sigma_full.sigma.replace(0, np.nan)
sigma_full["sigma"] = sigma_full.sigma.fillna(sigma_full.venta_prom_sku.fillna(0)*cv_mediano).fillna(1.0)
sigma_full["origen_sigma"] = np.where(falta, "fallback CV", "propio")
sigma_full = sigma_full.rename(columns={"sigma":"rmse"})   # el modelo espera 'rmse'
print("\nOrigen del σ:", sigma_full.origen_sigma.value_counts().to_dict())
print(sigma_full[["sku","rmse","origen_sigma"]].head(8).to_string(index=False))


✓ Consumo real: 60 SKU · 844,903 u

Sesgo agregado al lag 6: +183.6%  (sección 10 reporta +23,2 %)
σ propio calculado para 48 SKU (≥6 meses de historia)
  σ desesgado vs. RMSE crudo: reducción de 61% en la escala del SS
  CV mediano (σ/venta) = 0.58 → base del fallback

Origen del σ: {'propio': 124, 'fallback CV': 14}
      sku       rmse origen_sigma
789000501   0.000000  fallback CV
789000046 323.684520       propio
789000050 313.384207       propio
789000048 136.246593       propio
789000030  92.863382       propio
789000047  66.327770       propio
789000049  97.380380       propio
789000233 159.366641       propio


/tmp/ipykernel_2628/2353219727.py:34: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: (g.fcst_u.sum()/g.venta_u.sum() - 1) if g.venta_u.sum()>0 else np.nan)
/tmp/ipykernel_2628/2353219727.py:56: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: float(np.sqrt(((g.venta_u-g.fcst_u)**2).mean())))


In [14]:
###=== CELDA 12 [CÓDIGO] ===###
# --- F. Vencimientos V_{p,t} ---
venc_long = bq_query(f"""
SELECT CAST(item_no AS STRING) AS sku,
       DATE_TRUNC(DATE(expiration_date), MONTH) AS mes,
       SUM(SAFE_CAST(cantidad_vencida AS FLOAT64)) AS venc_u
FROM {TBL_VENC}
WHERE es_ultima_version = TRUE AND SAFE_CAST(cantidad_vencida AS FLOAT64) > 0
GROUP BY sku, mes
""")
venc_long["mes"] = pd.to_datetime(venc_long.mes)
venc_long = venc_long[(venc_long.mes >= anchor_ts) &
                      (venc_long.mes < anchor_ts + pd.DateOffset(months=T)) &
                      (venc_long.sku.isin(importados))]
print(f"✓ Vencimientos en el horizonte: {venc_long.venc_u.sum():,.0f} u")

# --- G. Inventario inicial I_{p,0} + verificación del balance ---
proj = bq_query(f"""
SELECT CAST(item_no AS STRING) AS sku,
       DATE(SAFE_CAST(year AS INT64), SAFE_CAST(month AS INT64), 1) AS mes,
       SUM(SAFE_CAST(inventory_initial AS FLOAT64)) AS inv_ini,
       SUM(SAFE_CAST(consumption AS FLOAT64)) AS consumo,
       SUM(SAFE_CAST(arrivals AS FLOAT64)) AS arribos,
       SUM(SAFE_CAST(expiration AS FLOAT64)) AS vencimiento,
       SUM(SAFE_CAST(inventory_final AS FLOAT64)) AS inv_fin
FROM {TBL_INV} WHERE es_ultima_version = TRUE
GROUP BY sku, mes
""")
proj["mes"] = pd.to_datetime(proj.mes)
proj["residuo"] = proj.inv_ini + proj.arribos - proj.consumo - proj.vencimiento - proj.inv_fin
malos = proj[proj.residuo.abs() > 1.0]
print(f"Balance (Ec. 8): residuo máx = {proj.residuo.abs().max():,.2f} u · "
      f"filas fuera de tolerancia: {len(malos):,}")
if len(malos): display(malos.head())

inv0 = proj[proj.mes == anchor_ts][["sku","inv_ini"]].rename(columns={"inv_ini":"inv0"})
inv0 = inv0[inv0.sku.isin(importados)]
inv0["inv0"] = inv0.inv0.clip(lower=0)
print(f"✓ I_p,0: {len(inv0)} SKU · {inv0.inv0.sum():,.0f} u")


✓ Vencimientos en el horizonte: 47,553 u
Balance (Ec. 8): residuo máx = 0.00 u · filas fuera de tolerancia: 0
✓ I_p,0: 61 SKU · 164,949 u


In [15]:
###=== CELDA 13 [CÓDIGO] ===###
# --- H. Costos (items) + I. Precios con IVA ---
items = bq_query(f"""
SELECT CAST(`No` AS STRING) AS sku,
       SAFE_CAST(Unit_Cost AS FLOAT64) AS unit_cost,
       SAFE_CAST(Unit_Price AS FLOAT64) / {IVA} AS precio_neto,  -- INCLUYE IVA
       CAST(Vendor_No AS STRING) AS vendor_no, Inactive AS inactive
FROM {TBL_ITEMS} WHERE `No` IS NOT NULL
""")
items = items[~items.inactive.astype(str).str.upper().isin(["TRUE","1","YES"])]
print(f"✓ Items activos: {len(items):,}")

# --- Tabla de PARÁMETROS por SKU ---
param = (maestro[["sku","descripcion","hfb","units_per_pallet","units_per_case",
                  "container_group"]]
         .merge(items[["sku","unit_cost"]], on="sku", how="inner")
         .merge(sigma_full[["sku","rmse"]], on="sku", how="left")
         .merge(shelf[["sku","shelf_life_w","shelf_life_p25"]], on="sku", how="left"))

# ABC bidimensional (valor = consumo × costo · volumen = consumo)
cons = ventas_long.groupby("sku", as_index=False).venta_u.sum().rename(columns={"venta_u":"vol"})
cons = cons.merge(items[["sku","unit_cost"]], on="sku", how="left")
cons["valor"] = cons.vol * cons.unit_cost
def pareto(df, col):
    d = df.sort_values(col, ascending=False)
    cum = d[col].cumsum() / d[col].sum()
    return pd.Series(np.where(cum <= .80, "A", np.where(cum <= .95, "B", "C")), index=d.index)
cons["c_val"] = pareto(cons, "valor"); cons["c_vol"] = pareto(cons, "vol")
rk = {"A":3,"B":2,"C":1}; iv = {3:"A",2:"B",1:"C"}
cons["abc_class"] = cons.apply(lambda r: iv[max(rk[r.c_val], rk[r.c_vol])], axis=1)
param = param.merge(cons[["sku","abc_class"]], on="sku", how="left")

param["abc_class"] = param.abc_class.fillna("C")
param["rmse"] = param.rmse.fillna(1185.0)
med = param.groupby("container_group").shelf_life_w.transform("median")
param["vu_supuesta"] = param.shelf_life_w.isna()
param["shelf_life_w"] = param.shelf_life_w.fillna(med).fillna(6)
param["shelf_life_p25"] = param.shelf_life_p25.fillna(param.shelf_life_w)
param["expiry_cost"] = param.unit_cost

assert (param.units_per_pallet > 0).all() and (param.unit_cost > 0).all()
print(f"✓ Parámetros: {len(param)} SKU · ABC {param.abc_class.value_counts().to_dict()}")


✓ Items activos: 229
✓ Parámetros: 383 SKU · ABC {'A': 235, 'C': 85, 'B': 63}


In [ ]:
#=== CELDA 14 [MARKDOWN] ===#
## Ensamblado y ejecución del MILP

Se construyen los vectores por SKU y se resuelve el modelo en **dos escenarios de vida útil**.

**Cola técnica:** la restricción de vida útil mira hacia adelante. Para no bloquear
artificialmente los pedidos del final del horizonte, la demanda se extiende con
*seasonal naive*. La cola **no es un pronóstico**, es un supuesto técnico documentado.
---

In [16]:
###=== CELDA 15 [CÓDIGO] — con deduplicación ===###
meses = pd.date_range(ANCHOR, periods=T, freq="MS").normalize()
def pivot(df, col):
    d = df.copy(); d["mes"] = pd.to_datetime(d.mes).dt.to_period("M").dt.to_timestamp()
    w = d.pivot_table(index="sku", columns="mes", values=col, aggfunc="sum").fillna(0.0)
    return {s: w.reindex(columns=meses, fill_value=0.0).loc[s].to_numpy(float) for s in w.index}

forecast_raw = pivot(fcst_long, "forecast_u")
expirations  = pivot(venc_long, "venc_u")
arrivals     = pivot(arr_long,  "arr_u")
inv0_map     = dict(zip(inv0.sku, inv0.inv0))
hist_mes     = ventas_long.pivot_table(index="sku", columns="mes", values="venta_u", aggfunc="sum")

# ---- DEDUPLICACIÓN: un SKU = una fila ----
# Criterio: se conserva la fila con mayor units_per_pallet informado (evita
# quedarse con un registro incompleto). Ajustar si la fuente indica otro criterio.
n_antes = len(param)
param = (param.sort_values(["sku","units_per_pallet"], ascending=[True, False])
              .drop_duplicates(subset="sku", keep="first")
              .reset_index(drop=True))
if n_antes != len(param):
    print(f"⚠️  Deduplicación: {n_antes:,} → {len(param):,} filas ({n_antes-len(param)} eliminadas)")

MESES_COLA = L
T_EXT = T + MESES_COLA

skus, forecast, expir, arriv = [], {}, {}, {}
vistos = set()
for _, r in param[param.sku.isin(forecast_raw)].iterrows():
    s = r.sku
    if s in vistos:            # salvaguarda extra
        continue
    vistos.add(s)
    h = hist_mes.loc[s].dropna() if s in hist_mes.index else None
    forecast[s] = extend_demand(forecast_raw[s], h, meses_extra=MESES_COLA)
    expir[s] = np.concatenate([expirations.get(s, np.zeros(T)), np.zeros(MESES_COLA)])
    arriv[s] = np.concatenate([arrivals.get(s, np.zeros(T)), np.zeros(MESES_COLA)])
    skus.append(SKU(
        sku_id=s, abc_class=r.abc_class,
        units_per_pallet=int(r.units_per_pallet), units_per_case=float(r.units_per_case),
        shelf_life_months=int(r.shelf_life_w),
        shelf_life_w=int(r.shelf_life_w), shelf_life_p25=int(r.shelf_life_p25),
        unit_cost=float(r.unit_cost), expiry_cost=float(r.expiry_cost),
        stockout_penalty=0.0, rmse=float(r.rmse),
        init_inventory=float(inv0_map.get(s, 0.0)),
        container_group=r.container_group))

assert len(skus) == len({s.sku_id for s in skus}), "Aún hay SKU duplicados"
print(f"✓ {len(skus)} SKU únicos · horizonte {T_EXT} ({T} forecast + {MESES_COLA} cola)")
print(f"  grupos: {pd.Series([s.container_group for s in skus]).value_counts().to_dict()}")

⚠️  Deduplicación: 383 → 59 filas (324 eliminadas)
✓ 54 SKU únicos · horizonte 18 (12 forecast + 6 cola)
  grupos: {'SE-SECO': 32, 'SE-CONGELADO': 22}


In [17]:
###=== CELDA 16 [CÓDIGO] ===###
cfg = ModelConfig(horizon=T_EXT, lead_time=L, forecast_bias=BIAS,
                  objetivo="stock",          # minimiza pallet-mes sujeto a servicio
                  uf_clp=UF_CLP, tarifa_uf_pallet_dia=TARIFA_UF_PALLET_DIA,
                  meses_cola=MESES_COLA, time_limit=300, gap_rel=0.01)

todos = solve_scenarios(skus, forecast, expir, arriv, cfg)

for nom, r in todos.items():
    d = r["resultados"]
    print(f"{nom:12s} {r['status']:10s} pallet-mes={d.pallets_bodega.sum():>6} "
          f"quiebre={int(d.quiebre_u.sum()):>9,} u")

BASE         Not Solved pallet-mes=  1548 quiebre=  159,478 u
CONSERVADOR  Optimal    pallet-mes=  2948 quiebre=  179,827 u


In [18]:
##=== CELDA 17 [CÓDIGO] ===###
# --- EL PEDIDO DE ESTE MES (lo único que se ejecuta) ---
for nom, r in todos.items():
    print("="*78); print(f"ESCENARIO {nom}")
    ped = pedido_del_mes(r)
    display(ped)

# Excel para Compras (celdas amarillas = ajustables antes de cargar al proveedor)
xlsx = export_excel(todos["BASE"], "pedido_del_mes.xlsx", results_all=todos)
from google.colab import files; files.download(xlsx)


ESCENARIO BASE
PEDIDO A EJECUTAR ESTE MES
  arribo estimado: en 6 meses (liberación SEREMI)
  SE-CONGELADO         94 pallets → 4×estiba 23
  SE-SECO              66 pallets → 3×estiba 23


,sku,descripcion,clase_abc,grupo,pallets,cajas,unidades_base,meses_de_venta,mes_arribo_estimado,contenedor
0,789000000,,B,SE-CONGELADO,2,127.6,1276,2.97,7,4×estiba 23
1,789000004,,A,SE-CONGELADO,2,72.7,1091,3.89,7,4×estiba 23
2,789000011,,A,SE-CONGELADO,2,39.0,935,2.99,7,4×estiba 23
3,789000012,,B,SE-CONGELADO,1,28.3,679,2.06,7,4×estiba 23
4,789000014,,A,SE-CONGELADO,4,155.5,2799,3.95,7,4×estiba 23
5,789000015,,A,SE-CONGELADO,1,27.6,496,4.25,7,4×estiba 23
12,789000029,,C,SE-CONGELADO,2,68.2,737,1.51,7,4×estiba 23
14,789000033,,B,SE-CONGELADO,3,139.0,973,1.43,7,4×estiba 23
15,789000039,,A,SE-CONGELADO,15,583.6,10589,1.39,7,4×estiba 23
19,789000045,,B,SE-CONGELADO,2,71.0,568,2.28,7,4×estiba 23


ESCENARIO CONSERVADOR
PEDIDO A EJECUTAR ESTE MES
  arribo estimado: en 6 meses (liberación SEREMI)
  SE-CONGELADO        115 pallets → 5×estiba 23
  SE-SECO              92 pallets → 4×estiba 23


,sku,descripcion,clase_abc,grupo,pallets,cajas,unidades_base,meses_de_venta,mes_arribo_estimado,contenedor
0,789000000,,B,SE-CONGELADO,3,189.0,1890,4.40,7,5×estiba 23
1,789000004,,A,SE-CONGELADO,1,40.0,600,2.14,7,5×estiba 23
2,789000005,,C,SE-CONGELADO,1,24.0,576,4.65,7,5×estiba 23
3,789000011,,A,SE-CONGELADO,3,72.0,1728,5.53,7,5×estiba 23
4,789000012,,B,SE-CONGELADO,1,24.0,576,1.74,7,5×estiba 23
5,789000014,,A,SE-CONGELADO,1,40.0,720,1.02,7,5×estiba 23
6,789000015,,A,SE-CONGELADO,3,60.0,1080,9.25,7,5×estiba 23
14,789000029,,C,SE-CONGELADO,3,120.0,1296,2.66,7,5×estiba 23
16,789000033,,B,SE-CONGELADO,2,90.0,630,0.92,7,5×estiba 23
17,789000039,,A,SE-CONGELADO,28,1118.8,20300,2.66,7,5×estiba 23


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [19]:
###=== CELDA 18 [CÓDIGO] ===###
# --- VALORIZACIÓN DEL AHORRO (línea base = presupuesto real) ---
TARIFA_PALLET_MES = TARIFA_UF_PALLET_DIA * DIAS_MES * UF_CLP

def valorizar(r, nombre):
    d = r["resultados"]
    d12 = d[d.periodo <= T]                       # solo el horizonte real, sin la cola
    pm = int(d12.pallets_bodega.sum())
    gasto = pm * TARIFA_PALLET_MES
    merma = float(d12.merge(param[["sku","unit_cost"]], on="sku")
                     .eval("vencimiento_u * unit_cost").sum())
    print(f"\n{nombre}")
    print(f"  Bodega base (real)  : ${GASTO_BASE_ANUAL:>14,.0f}  "
          f"({GASTO_BASE_ANUAL/TARIFA_PALLET_MES:,.0f} pallet-mes)")
    print(f"  Bodega modelo       : ${gasto:>14,.0f}  ({pm:,} pallet-mes)")
    print(f"  AHORRO              : ${GASTO_BASE_ANUAL-gasto:>14,.0f}  "
          f"({(1-gasto/GASTO_BASE_ANUAL)*100:.1f} %)")
    print(f"  Merma proyectada    : {int(d12.vencimiento_u.sum()):,} u = ${merma:,.0f}")
    print(f"  Quiebre proyectado  : {int(d12.quiebre_u.sum()):,} u")

for nom, r in todos.items():
    valorizar(r, f"ESCENARIO {nom}")



ESCENARIO BASE
  Bodega base (real)  : $   158,515,230  (422 pallet-mes)
  Bodega modelo       : $   440,293,669  (1,173 pallet-mes)
  AHORRO              : $  -281,778,439  (-177.8 %)
  Merma proyectada    : 43,845 u = $101,616,443
  Quiebre proyectado  : 159,478 u

ESCENARIO CONSERVADOR
  Bodega base (real)  : $   158,515,230  (422 pallet-mes)
  Bodega modelo       : $   737,576,351  (1,965 pallet-mes)
  AHORRO              : $  -579,061,121  (-365.3 %)
  Merma proyectada    : 43,845 u = $101,616,443
  Quiebre proyectado  : 172,929 u


In [ ]:
###== CELDA SS [MARKDOWN] — Análisis del stock de seguridad ===###
## Análisis de stock de seguridad y su tensión con la vida útil

El stock de seguridad es inventario **permanente** en bodega. Si supera lo que el
producto alcanza a rotar dentro de su vida útil residual, el propio SS **garantiza
vencimientos**: obliga a comprar producto que caducará antes de venderse.

Esta celda construye la tabla de evaluación y la escribe en BigQuery, comparando:
- **SS actual** (Z según clase ABC) vs. **SS propuesto** (nivel de servicio mayor)
- Cada uno expresado en **unidades** y en **meses de venta** del último forecast
- El **techo por vida útil**: SS máximo que rota sin vencer
- Una **bandera** para los SKU donde el SS excede ese techo

In [21]:
###=== CELDA SS [CÓDIGO] — tabla de stock de seguridad → BigQuery ===###
from scipy.stats import norm   # si falla: !pip -q install scipy

# --- Escenarios de nivel de servicio a comparar ---
NS_ACTUAL   = {"A": 0.98, "B": 0.95, "C": 0.90}     # vigente
NS_PROPUESTO= {"A": 0.99, "B": 0.98, "C": 0.95}     # ⚠️ ajustar al nivel deseado
FRACCION_VU = 0.5    # el SS no debe superar esta fracción de lo consumible en su vida útil

z = lambda ns: float(norm.ppf(ns))

# demanda mensual del ÚLTIMO forecast emitido (ya desesgada, como la usa el modelo)
dem_mes = {s: float(np.mean(v)) for s, v in
           {sk.sku_id: forecast[sk.sku_id][:T] / (1 + BIAS) for sk in skus}.items()}

filas = []
for sk in skus:
    s   = sk.sku_id
    d   = dem_mes.get(s, 0.0)
    VU  = int(sk.shelf_life_months)
    sg  = float(sk.rmse)
    ns_a, ns_p = NS_ACTUAL[sk.abc_class], NS_PROPUESTO[sk.abc_class]
    ss_a = z(ns_a) * sg * np.sqrt(L)
    ss_p = z(ns_p) * sg * np.sqrt(L)
    # techo por vida útil: lo que rota dentro de VU sin vencer
    consumible_vu = d * VU
    techo_vu = FRACCION_VU * consumible_vu
    filas.append({
        "sku": s,
        "descripcion": getattr(sk, "descripcion", ""),
        "clase_abc": sk.abc_class,
        "grupo": sk.container_group,
        "vida_util_meses": VU,
        "demanda_mensual_u": d,
        "sigma": sg,
        "lead_time_meses": L,
        "ns_actual": ns_a,       "z_actual": z(ns_a),
        "ss_actual_u": ss_a,     "ss_actual_meses": (ss_a/d if d>0 else np.nan),
        "ns_propuesto": ns_p,    "z_propuesto": z(ns_p),
        "ss_propuesto_u": ss_p,  "ss_propuesto_meses": (ss_p/d if d>0 else np.nan),
        "delta_u": ss_p - ss_a,
        "delta_pct": (ss_p/ss_a - 1) if ss_a > 0 else np.nan,
        "techo_vida_util_u": techo_vu,
        "excede_vida_util_actual":    bool(ss_a > techo_vu and techo_vu > 0),
        "excede_vida_util_propuesto": bool(ss_p > techo_vu and techo_vu > 0),
        "ss_aplicable_u": min(ss_p, techo_vu) if techo_vu > 0 else ss_p,
    })
ss_tbl = pd.DataFrame(filas)
ss_tbl["pallets_actual"]    = ss_tbl.ss_actual_u    / [sk.units_per_pallet for sk in skus]
ss_tbl["pallets_propuesto"] = ss_tbl.ss_propuesto_u / [sk.units_per_pallet for sk in skus]

print("RESUMEN POR CLASE ABC")
print(ss_tbl.groupby("clase_abc").agg(
    n_sku=("sku","count"),
    ss_actual_u=("ss_actual_u","sum"),
    ss_propuesto_u=("ss_propuesto_u","sum"),
    meses_actual=("ss_actual_meses","median"),
    meses_propuesto=("ss_propuesto_meses","median"),
    excede_VU_actual=("excede_vida_util_actual","sum"),
    excede_VU_prop=("excede_vida_util_propuesto","sum")).round(1).to_string())

print("\nRESUMEN POR VIDA ÚTIL")
ss_tbl["rango_vu"] = pd.cut(ss_tbl.vida_util_meses, [0,3,6,9,12,99],
                            labels=["≤3m","4-6m","7-9m","10-12m",">12m"])
print(ss_tbl.groupby("rango_vu", observed=True).agg(
    n_sku=("sku","count"),
    ss_prop_meses=("ss_propuesto_meses","median"),
    excede_VU=("excede_vida_util_propuesto","sum")).round(1).to_string())

n_riesgo = int(ss_tbl.excede_vida_util_propuesto.sum())
print(f"\n⚠️  {n_riesgo} SKU donde el SS propuesto EXCEDE lo consumible en su vida útil.")
print("    En esos casos el stock de seguridad obligaría a comprar producto que vencerá.")
if n_riesgo:
    print(ss_tbl[ss_tbl.excede_vida_util_propuesto].nlargest(10,"ss_propuesto_u")[
        ["sku","clase_abc","vida_util_meses","demanda_mensual_u",
         "ss_propuesto_u","ss_propuesto_meses","techo_vida_util_u","ss_aplicable_u"]
    ].round(1).to_string(index=False))

RESUMEN POR CLASE ABC
           n_sku  ss_actual_u  ss_propuesto_u  meses_actual  meses_propuesto  excede_VU_actual  excede_VU_prop
clase_abc                                                                                                     
A             18     121650.6        137797.5           3.0              3.4                 9              10
B             16      14655.7         18299.0           1.2              1.5                 3               4
C             20       6318.8          8110.2           1.7              2.2                 5               6

RESUMEN POR VIDA ÚTIL
          n_sku  ss_prop_meses  excede_VU
rango_vu                                 
≤3m           1            2.2          1
4-6m         28            2.3         13
7-9m         10            3.9          5
10-12m       10            1.7          1
>12m          5            2.6          0

⚠️  20 SKU donde el SS propuesto EXCEDE lo consumible en su vida útil.
    En esos casos el stock de segu

In [22]:
###=== CELDA SS2 [CÓDIGO] — escribir la tabla en BigQuery ===###
DATASET_DESTINO = "Availability_FOOD_CL"          # ⚠️ ajustar si corresponde
TABLA_DESTINO   = "stock_seguridad_evaluacion"

from google.cloud import bigquery as bq
out = ss_tbl.copy()
out["fecha_calculo"] = pd.Timestamp.now().normalize()
out["anchor"]        = pd.Timestamp(ANCHOR)
out["version"]       = pd.Timestamp.now().strftime("%Y%m%d_%H%M")
out["rango_vu"]      = out.rango_vu.astype(str)

dest = f"{PROJECT}.{DATASET_DESTINO}.{TABLA_DESTINO}"
job = client.load_table_from_dataframe(
    out, dest,
    job_config=bq.LoadJobConfig(
        write_disposition="WRITE_APPEND",   # conserva histórico de versiones
        schema_update_options=[bq.SchemaUpdateOption.ALLOW_FIELD_ADDITION],
    ))
job.result()
print(f"✓ {len(out)} filas escritas en {dest}")
print(f"  versión: {out.version.iloc[0]}  ·  anchor: {ANCHOR}")

# Verificación
chk = bq_query(f"""
SELECT version, COUNT(*) AS filas,
       ROUND(SUM(ss_actual_u))    AS ss_actual_total,
       ROUND(SUM(ss_propuesto_u)) AS ss_propuesto_total,
       COUNTIF(excede_vida_util_propuesto) AS sku_en_riesgo
FROM `{dest}`
GROUP BY version ORDER BY version DESC LIMIT 5
""")
display(chk)

✓ 54 filas escritas en tc-sc-bi-bigdata-ikea-corp-qa.Availability_FOOD_CL.stock_seguridad_evaluacion
  versión: 20260720_0040  ·  anchor: 2026-07-01


,version,filas,ss_actual_total,ss_propuesto_total,sku_en_riesgo
0,20260720_0040,54,142625.0,164207.0,20
